# Vision Transformer (ViT) - Implemented from Scratch
## Emotion Recognition from Abstract Art
### Architecture follows Dosovitskiy et al. (2021) - "An Image is Worth 16x16 Words"

In [43]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torchvision.transforms as transforms
from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

In [44]:
# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [45]:
# Load data
df = pd.read_csv("../data/dataset_final.csv")
IMAGE_DIR = Path("../data/images")

# Label encoding
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])

# Splits
df_subset, _ = train_test_split(df, train_size=3000, random_state=42, stratify=df['label'])

train_df, temp_df = train_test_split(df_subset, test_size=0.2, random_state=42, stratify=df_subset['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])


# Class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 2400 | Val: 300 | Test: 300


In [46]:
# Patch Embedding
class PatchEmbedding(nn.Module):
    """
    Splits image into fixed-size patches and linearly projects them into embeddings.
    Input:  (B, 3, 224, 224)
    Output: (B, num_patches, embed_dim)
    """
    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)       # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)             # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)        # (B, num_patches, embed_dim)
        return x

In [47]:
# Multi-Head Self-Attention
class MultiHeadSelfAttention(nn.Module):
    """
    Standard multi-head self-attention mechanism.
    Each head learns different aspects of patch relationships.
    """
    def __init__(self, embed_dim=768, num_heads=12, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

In [ ]:
# Transformer Block
class TransformerBlock(nn.Module):
    """
    Single transformer encoder block:
    LayerNorm -> MHSA -> residual + LayerNorm -> MLP -> residual
    """
    def __init__(self, embed_dim=768, num_heads=12, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
# ViT Classifier
class ViTClassifier(nn.Module):
    """
    Full Vision Transformer for emotion classification.
    Follows Dosovitskiy et al. (2021):
    PatchEmbedding -> CLS token -> Positional Encoding -> 
    TransformerBlocks -> CLS output -> Classification head
    """
    def __init__(self, image_size=224, patch_size=16, in_channels=3,
                 num_classes=8, embed_dim=768, depth=6,
                 num_heads=12, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        # Weight initialisation
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)

        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_token, x], dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)

        x = self.blocks(x)
        x = self.norm(x)

        return self.head(x[:, 0])

In [50]:
# Verify Model 
model = ViTClassifier(
    image_size=224,
    patch_size=16,
    num_classes=8,
    embed_dim=256,    
    depth=4,         
    num_heads=8,    
    mlp_ratio=2,    
    dropout=0.1
).to(device)

# Test with a dummy batch
dummy = torch.randn(2, 3, 224, 224).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")  # Expected: torch.Size([2, 8])

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Output shape: torch.Size([2, 8])
Total parameters: 2,358,536


In [51]:
# DataLoaders
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class ArtEmisDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.image_dir / row['filename']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row['label'], dtype=torch.long)

train_loader = DataLoader(ArtEmisDataset(train_df, IMAGE_DIR, train_transforms), batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(ArtEmisDataset(val_df, IMAGE_DIR, val_test_transforms), batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(ArtEmisDataset(test_df, IMAGE_DIR, val_test_transforms), batch_size=32, shuffle=False, num_workers=0)

print("DataLoaders ready")

DataLoaders ready


In [52]:
# Training Loop
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

In [53]:
# Train
EPOCHS = 5
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "../data/best_vit_model.pth")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/5 | Train Loss: 2.2007 | Train Acc: 0.1187 | Val Loss: 2.1218 | Val Acc: 0.0767
Epoch 2/5 | Train Loss: 2.1401 | Train Acc: 0.1363 | Val Loss: 2.1164 | Val Acc: 0.1167
Epoch 3/5 | Train Loss: 2.1119 | Train Acc: 0.1629 | Val Loss: 2.0682 | Val Acc: 0.1900
Epoch 4/5 | Train Loss: 2.0939 | Train Acc: 0.1487 | Val Loss: 2.0835 | Val Acc: 0.2467
Epoch 5/5 | Train Loss: 2.0815 | Train Acc: 0.1350 | Val Loss: 2.0970 | Val Acc: 0.1533
